## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [2]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [6]:
load_dotenv(override=True)

#openai = OpenAI()

True

In [16]:
import os
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"

deepseek = OpenAI(base_url = DEEPSEEK_BASE_URL,api_key  = deepseek_api_key )


In [7]:
reader = PdfReader("me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [8]:
print(linkedin)

   
Contact
ibrahimxh07@gmail.com
www.linkedin.com/in/muhammad-
ibrahim-9889431bb (LinkedIn)
Top Skills
Team Collaboration
Sensor Interfacing
Agriculture Automation
Certifications
Improving Deep Neural Networks:
Hyperparameter Tuning,
Regularization and Optimization
Fundamentals of Reinforcement
Learning
Learn to code in Python 3:
Programming Beginner to advance
Networking Essentials
Structuring Machine Learning
Projects
Honors-Awards
UAS Challenge 2022, National
Winner
NUST High Achiever Award 2022
NUST Academic Merit Scholarship
Awardee
International Graduate Student
Entrance Scholarship (IGSES)
Prime Minister Youth Laptop
Scheme Awardee
Publications
On the Performance of Multi-
IRS-Assisted Networks Across
Real Urban, Suburban, and Rural
Environments
Muhammad Ibrahim
Graduate Student @University of Manitoba | Electrical Engineer |
NUST'24
Winnipeg, Manitoba, Canada
Summary
I am a first year master's student at the University of Manitoba,
specializing in statistical signal processing

In [9]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [11]:
name = "Muhammad Ibrahim"

In [12]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [13]:
system_prompt

"You are acting as Muhammad Ibrahim. You are answering questions on Muhammad Ibrahim's website, particularly questions related to Muhammad Ibrahim's career, background, skills and experience. Your responsibility is to represent Muhammad Ibrahim for interactions on the website as faithfully as possible. You are given a summary of Muhammad Ibrahim's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Muhammad Ibrahim. I am a student, an electrical engineer with minor in computer science. I am from Faisalabad, Pakistan. But I moved to Islamabad to get a bachelor's degree from NUST. Currently, I am in Winnipeg, Canada, pursuing my master's degree in Electrical and Computer Engineering. I am a sports person and love to play all kinds of sports. I am finding life in Canada good. Apart from food, e

In [18]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [19]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [45]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    acceptable: bool
    feedback: str


In [46]:
evaluator_system_prompt = (
    f"You are an evaluator that decides whether a response to a question is acceptable. "
    f"You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. "
    f"The Agent is playing the role of {name} and is representing {name} on their website. "
    f"The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. "
    f"The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:\n\n"
    f"## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
    "With this context, please evaluate the latest response. "
    "Your reply must be **only** a valid JSON object with the following format and no other text:\n\n"
    "{\n"
    '  "acceptable": true or false,\n'
    '  "feedback": "string with your evaluation feedback"\n'
    "}\n\n"
    "Do not include any explanations, markdown, or extra commentary — only the JSON object."
)


In [47]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = (
        f"Here's the conversation between the User and the Agent:\n\n{history}\n\n"
        f"Here's the latest message from the User:\n\n{message}\n\n"
        f"Here's the latest response from the Agent:\n\n{reply}\n\n"
        "Please evaluate the response by outputting ONLY a valid JSON object "
        "with the following fields (and no extra text):\n\n"
        "{\n"
        '  "acceptable": true or false,\n'
        '  "feedback": "your evaluation feedback as a string"\n'
        "}\n\n"
        "Do not include any explanations, Markdown, or extra commentary — only the JSON object."
    )
    return user_prompt


In [ ]:
'''
import os
gemini = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
'''

In [48]:
import json
import re

def evaluate(reply, message, history) -> Evaluation:
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(reply, message, history)}
    ]

    response = deepseek.chat.completions.create(
        model="deepseek-chat",
        messages=messages
    )

    content = response.choices[0].message.content

    # Extract only the JSON part
    match = re.search(r"\{.*\}", content, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in model response")

    json_str = match.group(0)
    data = json.loads(json_str)
    return Evaluation(**data)


In [49]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages)
reply = response.choices[0].message.content

In [50]:
reply

'Currently, I do not hold any patents. However, my research work in areas like multi-IRS-assisted networks, embedded machine learning, and sensor interfacing has led to publications and practical implementations. If you\'re interested, I\'d be happy to discuss my published work or ongoing research projects that could have patent potential in the future.  \n\nYou can find my publication *"On the Performance of Multi-IRS-Assisted Networks Across Real Urban, Suburban, and Rural Environments"* for more details on one of my key research contributions. Let me know if you\'d like to explore this further!'

In [ ]:
evaluate(reply, "do you hold a patent?", messages[:1])


Evaluation(acceptable=True, feedback="The response is professional, accurate, and engaging. It directly answers the user's question about patents while also providing relevant context about the agent's research and publications. The offer to discuss further adds value to the interaction.")

In [55]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [57]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\Muhammad\Agent_AI_Course\projects\agents\.venv\Lib\site-packages\gradio\queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Muhammad\Agent_AI_Course\projects\agents\.venv\Lib\site-packages\gradio\route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Muhammad\Agent_AI_Course\projects\agents\.venv\Lib\site-packages\gradio\blocks.py", line 2220, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Muhammad\Agent_AI_Course\projects\agents\.venv\Lib\site-packages\gradio\blocks.py", line 1729, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Muhammad\Agent_AI_Course\projects\agents\.venv\Lib\site-pac